# Z5008 Big Data Lab — Lab 07: Apache Kafka
## Event-Driven Mobile Money Analytics

**Course:** Z5008 Big Data Lab | MTech Data Science & AI  
**Institution:** IIT Madras Zanzibar  
**Instructor:** Dr. Innocent Nyalala  

---

### Learning Objectives
By the end of this lab you will be able to:
1. Create and manage Kafka topics using the AdminClient API
2. Implement a Kafka producer that streams structured JSON events
3. Build a Kafka consumer that computes real-time analytics
4. Process a Kafka stream with Spark Structured Streaming and windowed aggregations

---

### Docker Environment
| Service | Container | Endpoint |
|---------|-----------|----------|
| JupyterLab | jupyter | http://localhost:8888 (token: bigdata) |
| Kafka Broker | kafka | kafka:9092 |
| Kafka UI | kafka-ui | http://localhost:8085 |
| Spark | jupyter (local mode) | local[4] |

---
## Part 1: SparkSession with Kafka Support

We initialise a SparkSession with the `spark-sql-kafka` connector package.  
This connector is required for **Spark Structured Streaming** to read from Kafka topics.

---
## Part 0: Install Required Python Libraries

We need the `kafka-python` library to interact with Kafka from Python.  
This cell installs it if not already present.

In [1]:
!pip install -q kafka-python
print("✓ kafka-python installed successfully")

✓ kafka-python installed successfully


In [2]:
from pyspark.sql import SparkSession
import sys

# Use local mode to avoid cluster connection issues
spark = (
    SparkSession.builder
    .appName("Lab07_Kafka_MobileMoney")
    .master("local[4]")  # Local mode with 4 cores
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("=" * 60)
print("Spark Session Initialized Successfully")
print("=" * 60)
print(f"Spark version : {spark.version}")
print(f"Spark master  : {spark.sparkContext.master}")
print(f"App name      : {spark.sparkContext.appName}")
print(f"Python version: {sys.version.split()[0]}")
print("=" * 60)

Spark Session Initialized Successfully
Spark version : 3.5.0
Spark master  : local[4]
App name      : Lab07_Kafka_MobileMoney
Python version: 3.11.8


---
## Part 2: Kafka AdminClient — Create Topic

We use the `kafka-python` library's `KafkaAdminClient` to programmatically create the `lab07_events` topic with **4 partitions** and replication factor 1.

**Why 4 partitions?**  
Partitions are the unit of parallelism in Kafka. With 4 partitions, up to 4 consumers in the same consumer group can read in parallel. Since our Spark cluster has 4 executor cores, this aligns well for maximum throughput.

In [3]:
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import TopicAlreadyExistsError

KAFKA_BOOTSTRAP = "kafka:9092"
TOPIC_NAME = "lab07_events"
NUM_PARTITIONS = 4
REPLICATION_FACTOR = 1

admin_client = KafkaAdminClient(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    client_id="lab07_admin"
)

new_topic = NewTopic(
    name=TOPIC_NAME,
    num_partitions=NUM_PARTITIONS,
    replication_factor=REPLICATION_FACTOR
)

try:
    admin_client.create_topics(new_topics=[new_topic], validate_only=False)
    print(f"Topic '{TOPIC_NAME}' created successfully.")
    print(f"  Partitions        : {NUM_PARTITIONS}")
    print(f"  Replication factor: {REPLICATION_FACTOR}")
except TopicAlreadyExistsError:
    print(f"Topic '{TOPIC_NAME}' already exists — skipping creation.")
finally:
    admin_client.close()

Topic 'lab07_events' already exists — skipping creation.


In [4]:
# Verify topic metadata
from kafka import KafkaConsumer

verify_consumer = KafkaConsumer(bootstrap_servers=KAFKA_BOOTSTRAP)
topics = verify_consumer.topics()
verify_consumer.close()

print("Available Kafka topics:")
for t in sorted(topics):
    marker = "  <-- lab07_events" if t == TOPIC_NAME else ""
    print(f"  {t}{marker}")

assert TOPIC_NAME in topics, f"ERROR: Topic '{TOPIC_NAME}' not found!"
print(f"\nVerified: '{TOPIC_NAME}' is present.")

Available Kafka topics:
  lab07_events  <-- lab07_events

Verified: 'lab07_events' is present.


---
## Part 3: KafkaProducer — Send 2000 Mobile Money Events

We generate realistic synthetic mobile money transactions for East Africa and send them to `lab07_events`.  
Each event is a JSON-serialised dictionary with the following schema:

```json
{
  "transaction_id" : "uuid4",
  "customer_id"    : "uuid4",
  "country"        : "Kenya | Tanzania | Uganda | Rwanda | Ethiopia",
  "amount_ksh"     : 150.0,
  "tx_type"        : "SEND | RECEIVE | DEPOSIT | WITHDRAW",
  "timestamp"      : "2026-03-21T10:15:00Z",
  "status"         : "SUCCESS | FAILED"
}
```

The **message key** is set to `country` so that all transactions from the same country are routed to the same partition, enabling ordered processing per country.

In [5]:
import json
import uuid
import random
from datetime import datetime, timezone, timedelta
from kafka import KafkaProducer

# Configuration
NUM_EVENTS = 2000
COUNTRIES = ["Kenya", "Tanzania", "Uganda", "Rwanda", "Ethiopia"]
# Weighted distribution matching East Africa transaction volumes
COUNTRY_WEIGHTS = [0.40, 0.25, 0.15, 0.10, 0.10]
TX_TYPES = ["SEND", "RECEIVE", "DEPOSIT", "WITHDRAW"]
TX_WEIGHTS = [0.35, 0.30, 0.20, 0.15]
STATUS_WEIGHTS = [0.95, 0.05]  # 95% SUCCESS

# Country-specific amount ranges (KES equivalent)
AMOUNT_RANGES = {
    "Kenya"   : (50, 150_000),
    "Tanzania": (50, 100_000),
    "Uganda"  : (50,  80_000),
    "Rwanda"  : (50,  60_000),
    "Ethiopia": (50,  50_000),
}

def generate_event() -> dict:
    country = random.choices(COUNTRIES, weights=COUNTRY_WEIGHTS)[0]
    lo, hi = AMOUNT_RANGES[country]
    # FIX: Use CURRENT timestamp for real-time streaming (not past dates)
    event_time = datetime.now(timezone.utc)
    return {
        "transaction_id": str(uuid.uuid4()),
        "customer_id"   : str(uuid.uuid4()),
        "country"       : country,
        "amount_ksh"    : round(random.uniform(lo, hi), 2),
        "tx_type"       : random.choices(TX_TYPES, weights=TX_WEIGHTS)[0],
        "timestamp"     : event_time.strftime("%Y-%m-%dT%H:%M:%SZ"),
        "status"        : random.choices(["SUCCESS", "FAILED"], weights=STATUS_WEIGHTS)[0],
    }

# Demonstrate the schema with one sample event
sample = generate_event()
print("Sample event:")
print(json.dumps(sample, indent=2))

Sample event:
{
  "transaction_id": "38dd7d31-4123-4653-9313-02e06c4c9c11",
  "customer_id": "46f65829-9cc6-4274-85f4-9deef5b878f9",
  "country": "Kenya",
  "amount_ksh": 116844.16,
  "tx_type": "SEND",
  "timestamp": "2026-06-01T12:45:18Z",
  "status": "SUCCESS"
}


In [6]:
producer = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8"),
    acks="all",           # wait for all in-sync replicas to acknowledge
    retries=3,
    max_block_ms=10_000,
)

sent = 0
errors = 0
country_counts = {c: 0 for c in COUNTRIES}

for i in range(NUM_EVENTS):
    event = generate_event()
    try:
        producer.send(
            TOPIC_NAME,
            key=event["country"],
            value=event
        )
        country_counts[event["country"]] += 1
        sent += 1
    except Exception as exc:
        print(f"Send error at event {i}: {exc}")
        errors += 1

    if (i + 1) % 500 == 0:
        print(f"  Progress: {i + 1}/{NUM_EVENTS} events sent...")

producer.flush()
producer.close()

print(f"\nProducer complete.")
print(f"  Sent   : {sent}")
print(f"  Errors : {errors}")
print("\nEvents by country:")
for country, count in sorted(country_counts.items(), key=lambda x: -x[1]):
    pct = count / sent * 100
    print(f"  {country:<12}: {count:>5} ({pct:.1f}%)")

  Progress: 500/2000 events sent...
  Progress: 1000/2000 events sent...
  Progress: 1500/2000 events sent...
  Progress: 2000/2000 events sent...

Producer complete.
  Sent   : 2000
  Errors : 0

Events by country:
  Kenya       :   806 (40.3%)
  Tanzania    :   517 (25.9%)
  Uganda      :   272 (13.6%)
  Ethiopia    :   206 (10.3%)
  Rwanda      :   199 (10.0%)


---
## Part 4: KafkaConsumer — Analytics by Country

We now consume all messages from `lab07_events` and compute statistics:
- **Count per country** — how many transactions came from each country
- **Total amount per country** — sum of `amount_ksh` grouped by country
- **Average amount per country** — mean transaction size

The consumer uses `auto_offset_reset='earliest'` to read all messages from the beginning.

In [7]:
from kafka import KafkaConsumer
from collections import defaultdict
import json

consumer = KafkaConsumer(
    TOPIC_NAME,
    bootstrap_servers=KAFKA_BOOTSTRAP,
    group_id="lab07_analytics_group",
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    value_deserializer=lambda v: json.loads(v.decode("utf-8")),
    consumer_timeout_ms=10_000,  # stop after 10 s of no messages
)

# Accumulators
count_by_country  = defaultdict(int)
amount_by_country = defaultdict(float)
all_amounts       = []
total_messages    = 0

try:
    for message in consumer:
        event = message.value
        country    = event.get("country", "Unknown")
        amount     = float(event.get("amount_ksh", 0))

        count_by_country[country]  += 1
        amount_by_country[country] += amount
        all_amounts.append(amount)
        total_messages += 1

except Exception as exc:
    print(f"Consumer error: {exc}")
finally:
    consumer.close()
    print("Consumer closed.")

print(f"\nTotal messages consumed: {total_messages}")

Consumer closed.

Total messages consumed: 2000


In [8]:
print("=" * 52)
print("         Mobile Money Consumer Analytics")
print("=" * 52)
print(f"Total messages consumed: {total_messages}")

print("\n--- Transaction Count by Country ---")
for country in sorted(count_by_country, key=lambda c: -count_by_country[c]):
    pct = count_by_country[country] / total_messages * 100 if total_messages > 0 else 0
    print(f"  {country:<12}: {count_by_country[country]:>6} ({pct:.1f}%)")

print("\n--- Total Amount by Country (KES) ---")
for country in sorted(amount_by_country, key=lambda c: -amount_by_country[c]):
    print(f"  {country:<12}: KES {amount_by_country[country]:>15,.2f}")

print("\n--- Average Amount by Country (KES) ---")
for country in sorted(count_by_country):
    avg = amount_by_country[country] / count_by_country[country] if count_by_country[country] > 0 else 0
    print(f"  {country:<12}: KES {avg:>12,.2f}")

# Percentiles via sorted list
if all_amounts:
    sorted_amounts = sorted(all_amounts)
    n = len(sorted_amounts)

    def percentile(data: list, p: float) -> float:
        idx = max(0, int(len(data) * p / 100) - 1)
        return data[idx]

    print("\n--- Amount Percentiles (all countries) ---")
    for pct in [50, 75, 90, 95, 99]:
        print(f"  P{pct:<3}: KES {percentile(sorted_amounts, pct):>12,.2f}")

print("=" * 52)

         Mobile Money Consumer Analytics
Total messages consumed: 2000

--- Transaction Count by Country ---
  Kenya       :    806 (40.3%)
  Tanzania    :    517 (25.9%)
  Uganda      :    272 (13.6%)
  Ethiopia    :    206 (10.3%)
  Rwanda      :    199 (10.0%)

--- Total Amount by Country (KES) ---
  Kenya       : KES   60,443,647.46
  Tanzania    : KES   25,324,087.57
  Uganda      : KES   11,117,730.89
  Rwanda      : KES    5,891,479.64
  Ethiopia    : KES    5,247,128.58

--- Average Amount by Country (KES) ---
  Ethiopia    : KES    25,471.50
  Kenya       : KES    74,992.12
  Rwanda      : KES    29,605.43
  Tanzania    : KES    48,982.76
  Uganda      : KES    40,874.01

--- Amount Percentiles (all countries) ---
  P50 : KES    47,258.57
  P75 : KES    78,499.68
  P90 : KES   110,694.32
  P95 : KES   130,509.22
  P99 : KES   145,817.71


---
## Part 5: Spark Structured Streaming from Kafka

We now use Spark Structured Streaming to process the `lab07_events` topic in real time.

### Pipeline Design
```
Kafka topic: lab07_events
         │
         ▼
  readStream (kafka source)
         │
         ▼
  parse JSON value → StructType
         │
         ▼
  watermark(30s) + tumbling_window(2min)
  groupBy(window, country)
  agg(count(), avg(amount_ksh))
         │
         ▼
  writeStream → console (update mode)
```

In [9]:
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)
from pyspark.sql.functions import (
    col, from_json, to_timestamp,
    window, count, avg, expr
)

# Define the JSON schema for the mobile money events
event_schema = StructType([
    StructField("transaction_id", StringType(),  nullable=False),
    StructField("customer_id",    StringType(),  nullable=True),
    StructField("country",        StringType(),  nullable=False),
    StructField("amount_ksh",     DoubleType(),  nullable=False),
    StructField("tx_type",        StringType(),  nullable=True),
    StructField("timestamp",      StringType(),  nullable=True),  # raw string
    StructField("status",         StringType(),  nullable=True),
])

print("Event schema defined:")
for field in event_schema.fields:
    print(f"  {field.name:<20} {field.dataType}")

Event schema defined:
  transaction_id       StringType()
  customer_id          StringType()
  country              StringType()
  amount_ksh           DoubleType()
  tx_type              StringType()
  timestamp            StringType()
  status               StringType()


In [10]:
# Read stream from Kafka
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", TOPIC_NAME)
    .option("startingOffsets", "earliest")
    .option("failOnDataLoss", "false")
    .load()
)

print("Raw stream schema:")
raw_stream.printSchema()

Raw stream schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [11]:
# Parse the Kafka value (binary) as JSON using the defined schema
parsed_stream = (
    raw_stream
    .select(
        from_json(
            col("value").cast("string"),
            event_schema
        ).alias("data")
    )
    .select("data.*")
    # Convert ISO 8601 string to TimestampType
    .withColumn(
        "event_ts",
        to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
    )
    .filter(col("transaction_id").isNotNull())
    .filter(col("amount_ksh") > 0)
)

print("Parsed stream schema:")
parsed_stream.printSchema()

Parsed stream schema:
root
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- amount_ksh: double (nullable = true)
 |-- tx_type: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- status: string (nullable = true)
 |-- event_ts: timestamp (nullable = true)



In [12]:
# Apply watermark and 2-minute tumbling window aggregation
windowed_agg = (
    parsed_stream
    .withWatermark("event_ts", "30 seconds")        # late-data tolerance
    .groupBy(
        window(col("event_ts"), "2 minutes"),        # tumbling window
        col("country")
    )
    .agg(
        count("transaction_id").alias("tx_count"),
        avg("amount_ksh").alias("avg_amount_ksh")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("country"),
        col("tx_count"),
        col("avg_amount_ksh")
    )
)

print("Windowed aggregation schema:")
windowed_agg.printSchema()

Windowed aggregation schema:
root
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)
 |-- country: string (nullable = true)
 |-- tx_count: long (nullable = false)
 |-- avg_amount_ksh: double (nullable = true)



In [13]:
# Write stream to console in update mode
query = (
    windowed_agg
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .option("numRows", 30)
    .trigger(processingTime="15 seconds")
    .queryName("lab07_kafka_window_agg")
    .start()
)

print(f"Streaming query started: {query.name}")
print(f"Query ID: {query.id}")
print("Waiting 60 seconds for micro-batches...")

query.awaitTermination(60)

print("Stream stopped after 60 seconds.")

# Print final status
status = query.lastProgress
if status:
    print(f"\nLast batch ID  : {status.get('batchId', 'N/A')}")
    print(f"Input rows/sec : {status.get('inputRowsPerSecond', 0):.2f}")
    print(f"Processed rows : {status.get('numInputRows', 0)}")

Streaming query started: lab07_kafka_window_agg
Query ID: e34b5311-efa2-43bc-aa59-fd2a3c015bb4
Waiting 60 seconds for micro-batches...


StreamingQueryException: [STREAM_FAILED] Query [id = e34b5311-efa2-43bc-aa59-fd2a3c015bb4, runId = 3c3cde84-129f-4a79-87e1-e10a799b1861] terminated with exception: Cannot find catalog plugin class for catalog 'spark_catalog': org.apache.spark.sql.delta.catalog.DeltaCatalog.

---
## Part 6: Real-Time Streaming Demo

Now we demonstrate **real-time streaming** by sending events continuously while Spark processes them in micro-batches.

Run the cell below in a **separate terminal** or run it WHILE the streaming query above is still running to see live updates.

In [ ]:
import time

# Real-time producer: send 100 events at 1 event/second
producer_rt = KafkaProducer(
    bootstrap_servers=KAFKA_BOOTSTRAP,
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
    key_serializer=lambda k: k.encode("utf-8"),
    acks="all",
)

print("Starting real-time event stream...")
print("=" * 60)

for i in range(100):
    event = generate_event()
    producer_rt.send(TOPIC_NAME, key=event["country"], value=event)
    
    if (i + 1) % 10 == 0:
        print(f"Sent {i + 1}/100 events | Country: {event['country']} | Amount: KES {event['amount_ksh']:,.2f}")
    
    time.sleep(1)  # 1 event per second for real-time demo

producer_rt.flush()
producer_rt.close()

print("=" * 60)
print("Real-time stream complete!")

---

## Visualizing Kafka with Kafka UI

Open **http://localhost:8085** in your browser to access Kafka UI.

You can:
- Browse the `lab07_events` topic
- View partition distribution
- Inspect individual messages (JSON format)
- Monitor consumer group offsets

This is a powerful tool for debugging Kafka applications in development!

### Explanation: Windowed Aggregation Results

The console output above shows Spark Structured Streaming micro-batch results:

- **`window_start` / `window_end`**: The 2-minute time window boundary. Each row represents transactions that occurred within that 2-minute interval.
- **`country`**: The country group within the window.
- **`tx_count`**: Number of transactions in this window+country combination.
- **`avg_amount_ksh`**: Mean transaction amount in KES for this window and country.

**Watermark behaviour**: The 30-second watermark means Spark will wait up to 30 seconds after the window closes before finalising results. Events arriving more than 30 seconds late for a given window are dropped. This trades completeness for bounded state size — essential for production streaming jobs.

**Update mode**: We use `update` output mode because we are aggregating with a watermark. In update mode, only rows that changed since the last micro-batch are written to the sink — this is more efficient than `complete` mode for large state.

In [ ]:
# Stop the SparkSession cleanly
spark.stop()
print("SparkSession stopped. Lab 07 complete.")

---
## Lab Summary

In this lab we covered the full Kafka pipeline:

| Step | Tool | Key Concept |
|------|------|-------------|
| 1 | SparkSession + kafka package | Connecting Spark to Kafka |
| 2 | KafkaAdminClient | Programmatic topic management |
| 3 | KafkaProducer | Event serialisation, partitioning by key |
| 4 | KafkaConsumer | Consumer groups, offset management, analytics |
| 5 | Spark Structured Streaming | readStream, watermark, tumbling window, writeStream |

**Next:** Assignment 7 extends this by sending 5,000 events, implementing graceful shutdown, and asserting correctness of the streaming output.